# High-latitude patch selection

Change `LATITUDE_CUT_DEG` below to select patches whose full square footprint lies above that absolute Galactic latitude.


In [13]:
from pathlib import Path
import csv
import re

import numpy as np
from astropy.coordinates import SkyCoord, SkyOffsetFrame
import astropy.units as u

# Resolve paths whether the notebook is launched from the repo root or from BICEP project/.
HERE = Path.cwd().resolve()
PROJECT_DIR = None
for base in (HERE, *HERE.parents):
    candidate = base / "BICEP project"
    if (candidate / "contents.txt").exists():
        PROJECT_DIR = candidate
        break
if PROJECT_DIR is None and HERE.name == "BICEP project" and (HERE / "contents.txt").exists():
    PROJECT_DIR = HERE
if PROJECT_DIR is None:
    raise FileNotFoundError("Could not find BICEP project/contents.txt from this working directory.")

CONTENTS_PATH = PROJECT_DIR / "contents.txt"

# Change this value as needed. The selection is for full patch footprints with |b| > LATITUDE_CUT_DEG.
LATITUDE_CUT_DEG = 45

PATCH_SIDE_PIXELS = 384
PIXEL_SCALE_ARCMIN = 3.4
HALF_SIDE_DEG = 0.5 * PATCH_SIDE_PIXELS * PIXEL_SCALE_ARCMIN / 60.0
N_BOUNDARY_SAMPLES = 256

PATCHES_CSV = PROJECT_DIR / "patches.csv"
IDS_TXT = PROJECT_DIR / "ids.txt"

print("Project directory:", PROJECT_DIR)
print("Latitude cut [deg]:", LATITUDE_CUT_DEG)
print("Patch half-side [deg]:", HALF_SIDE_DEG)


Project directory: /Users/tsouros/Desktop/Projects/STL-Dev/BICEP project
Latitude cut [deg]: 45
Patch half-side [deg]: 10.879999999999999


## Patch centers

If `contents.txt` contains raw Planck filenames, the centers are read from the filenames. If it contains only compsep outputs such as `p0_...npy`, the centers are reconstructed as HEALPix `nside=4`, ring-ordered patch centers.


In [14]:
RAW_PATCH_RE = re.compile(
    r"patch_(?P<patch_id>\d+)_.*?_cen_pix_lon_"
    r"(?P<lon>[-+0-9.eE]+)_lat_(?P<lat>[-+0-9.eE]+)_"
)
COMPSEP_PATCH_RE = re.compile(r"^p(?P<patch_id>\d+)_.*\.npy$")


def read_manifest(path):
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]


def centers_from_raw_filenames(lines):
    centers = {}
    for filename in lines:
        match = RAW_PATCH_RE.search(filename)
        if match is None:
            continue
        patch_id = int(match.group("patch_id"))
        lon = float(match.group("lon"))
        lat = float(match.group("lat"))
        previous = centers.get(patch_id)
        if previous is not None and not np.allclose(previous, (lon, lat)):
            raise ValueError(f"Patch {patch_id} has inconsistent centers: {previous} and {(lon, lat)}")
        centers[patch_id] = (lon, lat)
    return centers


def centers_from_healpix_ring(lines):
    patch_ids = sorted({
        int(match.group("patch_id"))
        for line in lines
        if (match := COMPSEP_PATCH_RE.match(line)) is not None
    })
    if not patch_ids:
        return {}
    try:
        import healpy as hp
    except ImportError as exc:
        raise ImportError(
            "contents.txt does not contain filename centers. Install/use a kernel with healpy, "
            "or provide raw filenames containing cen_pix_lon and lat."
        ) from exc

    nside = 4
    npix = hp.nside2npix(nside)
    theta, phi = hp.pix2ang(nside, np.arange(npix), nest=False)
    lon = np.degrees(phi) % 360.0
    lat = 90.0 - np.degrees(theta)
    return {patch_id: (float(lon[patch_id]), float(lat[patch_id])) for patch_id in patch_ids}


manifest_lines = read_manifest(CONTENTS_PATH)
patch_centers = centers_from_raw_filenames(manifest_lines)
center_source = "filename centers"
if not patch_centers:
    patch_centers = centers_from_healpix_ring(manifest_lines)
    center_source = "HEALPix nside=4 ring centers"
if not patch_centers:
    raise RuntimeError("No patch centers could be inferred from contents.txt")

print(f"Patch centers: {len(patch_centers)} ({center_source})")
print("First centers:", list(sorted(patch_centers.items()))[:5])


Patch centers: 192 (HEALPix nside=4 ring centers)
First centers: [(0, (45.0, 78.28414760510762)), (1, (135.0, 78.28414760510762)), (2, (225.0, 78.28414760510762)), (3, (315.0, 78.28414760510762)), (4, (22.5, 66.44353569089877))]


## Select patches


In [15]:
def sample_patch_boundary(lon_deg, lat_deg, half_side_deg, n_per_edge=N_BOUNDARY_SAMPLES):
    center = SkyCoord(l=lon_deg * u.deg, b=lat_deg * u.deg, frame="galactic")
    frame = SkyOffsetFrame(origin=center)
    t = np.linspace(-half_side_deg, half_side_deg, n_per_edge)
    x = np.concatenate([
        t,
        np.full_like(t, half_side_deg),
        t[::-1],
        np.full_like(t, -half_side_deg),
    ])
    y = np.concatenate([
        np.full_like(t, -half_side_deg),
        t,
        np.full_like(t, half_side_deg),
        t[::-1],
    ])
    boundary = SkyCoord(lon=x * u.deg, lat=y * u.deg, frame=frame).transform_to("galactic")
    return boundary.b.deg


rows = []
for patch_id, (lon, lat) in sorted(patch_centers.items()):
    boundary_b = sample_patch_boundary(lon, lat, HALF_SIDE_DEG)
    min_abs_b = float(np.min(np.abs(boundary_b)))
    selected = min_abs_b > LATITUDE_CUT_DEG
    rows.append({
        "patch_id": patch_id,
        "lon_deg": lon,
        "lat_deg": lat,
        "min_abs_b_deg": min_abs_b,
        "selected": selected,
    })

patch_ids = [row["patch_id"] for row in rows if row["selected"]]
print(f"Selected patches with full footprint |b| > {LATITUDE_CUT_DEG:g} deg: {len(patch_ids)}")
print(patch_ids)

with PATCHES_CSV.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

IDS_TXT.write_text("\n".join(str(patch_id) for patch_id in patch_ids) + "\n")
print("Wrote", PATCHES_CSV)
print("Wrote", IDS_TXT)


Selected patches with full footprint |b| > 45 deg: 24
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191]
Wrote /Users/tsouros/Desktop/Projects/STL-Dev/BICEP project/patches.csv
Wrote /Users/tsouros/Desktop/Projects/STL-Dev/BICEP project/ids.txt
